In [ ]:
import requests
import json
import time

auth = json.loads(mssparkutils.notebook.run("procore_auth"))
token = auth["token"]
COMPANY_ID = auth["company_id"]
headers = {"Authorization": f"Bearer {token}"}

print("Auth successful" if token else "Auth failed")

StatementMeta(, 72263f34-1708-47df-af79-308cb512f7c8, 3, Finished, Available, Finished, False)

Auth successful


In [2]:
projects_response = requests.get(
    "https://api.procore.com/rest/v1.0/projects",
    headers=headers,
    params={"company_id": COMPANY_ID}
)
projects = projects_response.json()
print(f"{len(projects)} projects found" if isinstance(projects, list) else "Failed to fetch projects")

StatementMeta(, 72263f34-1708-47df-af79-308cb512f7c8, 4, Finished, Available, Finished, False)

16 projects found


In [3]:
all_budget_rows = []

for project in projects:
    project_id = project["id"]
    project_name = project["name"]
    print(f"Pulling budget for: {project_name}")

    views_response = requests.get(
        "https://api.procore.com/rest/v1.0/budget_views",
        headers={**headers, "Procore-Company-Id": str(COMPANY_ID)},
        params={"project_id": project_id}
    )
    views = views_response.json()

    if not views or isinstance(views, dict):
        print(f"  No budget views found, skipping")
        continue

    budget_view = next(
        (v for v in views if v.get("role") == "budgeting"), 
        views[0]
    )
    budget_view_id = budget_view["id"]

    page = 1
    while True:
        rows_response = requests.get(
            f"https://api.procore.com/rest/v1.0/budget_views/{budget_view_id}/detail_rows",
            headers={**headers, "Procore-Company-Id": str(COMPANY_ID)},
            params={"project_id": project_id, "page": page, "per_page": 100}
        )
        rows = rows_response.json()

        if not rows or isinstance(rows, dict):
            break

        for row in rows:
            row["project_id"] = project_id
            row["project_name"] = project_name

        all_budget_rows.extend(rows)
        page += 1

    time.sleep(0.5)

print(f"Done! Total budget rows: {len(all_budget_rows)}")

StatementMeta(, 72263f34-1708-47df-af79-308cb512f7c8, 5, Finished, Available, Finished, False)

Pulling budget for: 1100 Fulton Street
Pulling budget for: 11 ESSEX ST
Pulling budget for: 337A & 337B West Broadway Rehabilitaion Work
Pulling budget for: 360 Lexington 8th & 20th Floor
Pulling budget for: 549 Munroe Av
Pulling budget for: 64 MET OVAL PSC + 1410 MET STOREROOM
Pulling budget for: Boys & Girls Club
Pulling budget for: EMBANKMENT PHASE II
Pulling budget for: Embankment + Revetment Apartments 270 & 310 10th Street NJ
Pulling budget for: Lillipvt 45 Renwick St
Pulling budget for: PCNA 711 11TH AVE
Pulling budget for: Sandbox Test Project
Pulling budget for: Standard Project Template
Pulling budget for: SYMRISE - 15th & 16th Flr
Pulling budget for: TEST - ABM SUBORDINATE
Pulling budget for: VOCO HOTEL TSQ
Done! Total budget rows: 334


In [4]:
import pandas as pd
import re
import json
from datetime import datetime

# Add snapshot date to every row
snapshot_date = datetime.today().strftime("%Y-%m-%d")

clean_rows = []
for row in all_budget_rows:
    clean_row = {}
    for key, value in row.items():
        if value is None:
            clean_row[key] = None
        elif isinstance(value, (dict, list)):
            clean_row[key] = json.dumps(value)
        elif isinstance(value, bool):
            clean_row[key] = str(value)
        elif isinstance(value, (int, float, str)):
            clean_row[key] = value
        else:
            clean_row[key] = str(value)
    clean_row["snapshot_date"] = snapshot_date
    clean_rows.append(clean_row)

def clean_column_name(col):
    col = col.strip()
    col = re.sub(r'[ ,;{}()\n\t=]', '_', col)
    col = re.sub(r'_+', '_', col)
    col = col.strip('_')
    return col

pdf = pd.DataFrame(clean_rows)
pdf.columns = [clean_column_name(c) for c in pdf.columns]

for col in pdf.columns:
    if pdf[col].dtype == object:
        pdf[col] = pdf[col].astype(str).replace('None', None)

# Drop existing table to clear old schema
spark.sql("DROP TABLE IF EXISTS procore_budgets_raw")

df = spark.createDataFrame(pdf)
df.write.format("delta").mode("append").saveAsTable("procore_budgets_raw")

print(f"Saved snapshot for {snapshot_date} to Bronze_Lakehouse successfully")

StatementMeta(, 72263f34-1708-47df-af79-308cb512f7c8, 6, Finished, Available, Finished, False)

Saved snapshot for 2026-05-06 to Bronze_Lakehouse successfully
